# Modeling and evaluation (Person 3)

CAP 3764, Fall 2026, Team 6. Rana Waleed Aftab (Person 3).

This notebook runs and explains `src/models.py`. It takes Person 2's model-ready split and preprocessor (`src/features.py`) and answers one question: **can a product's listing characteristics predict its average rating better than simply guessing the average?**

It covers:

1. Loading the training data (the test set stays untouched)
2. The four candidate models and why each is here
3. The validation design
4. 5-fold cross-validation results
5. What the results mean, and what comes next

Target: `average_rating` (1-5 stars), a regression problem. Metrics: RMSE and MAE (in stars, lower is better) and R² (share of variation explained, higher is better).

## 0. Setup

Run from the repo root so `src` imports work, and use the team's shared modules instead of re-implementing anything.

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import pandas as pd
from src import features as F
from src import models as M

pd.set_option("display.float_format", "{:.4f}".format)

## 1. Load the training data

`load_split()` reads the 80/20 split that Person 2's `python -m src.features` saved. We keep only the training part here. **The test set is not used in this notebook**: it is scored exactly once, after tuning, so that it gives an honest estimate of performance on unseen products.

In [2]:
X_train, _, y_train, _ = F.load_split()   # test set deliberately ignored
print(f"Training rows: {len(X_train):,}")
print("Features:", list(X_train.columns))

Training rows: 940,325
Features: ['log_price', 'log_rating_number', 'n_features', 'n_description', 'title_len', 'n_categories', 'n_details', 'has_price', 'price_is_from', 'category', 'main_category', 'store']


In [3]:
# Category mix of the training set (the split was stratified on this)
X_train["category"].value_counts(normalize=True).rename("share")

category
Toys_and_Games              0.5939
Industrial_and_Scientific   0.2242
Musical_Instruments         0.1167
All_Beauty                  0.0652
Name: share, dtype: Float64

## 2. The four models

Each model is wrapped in `Pipeline([("prep", make_preprocessor()), ("model", ...)])`. Because the preprocessor sits inside the pipeline, the imputer, scaler and encoders are re-fitted on the training folds of every CV split, so nothing from a validation fold leaks into preprocessing.

| Model | Why it's here |
|---|---|
| **Baseline** (`DummyRegressor`, mean) | Predicts the average training rating for every product. Any real model must beat it, otherwise the features carry no signal. |
| **Ridge** | Linear and regularized. Shows how much of the rating can be explained by straight-line relationships, and is easy to interpret. |
| **RandomForest** | Averages many decision trees, so it captures non-linear effects and interactions between features. Kept small (100 trees, 30% of rows per tree) to train in reasonable time. |
| **HistGradientBoosting** | Builds trees one after another, each correcting the last. Fast on hundreds of thousands of rows and usually the strongest model on tabular data. |

In [4]:
models = M.get_models()
for name, pipe in models.items():
    print(f"{name:<22} -> {pipe.named_steps['model']}")

Baseline (mean)        -> DummyRegressor()
Ridge                  -> Ridge()
RandomForest           -> RandomForestRegressor(max_samples=0.3, min_samples_leaf=5, n_jobs=-1,
                      random_state=42)
HistGradientBoosting   -> HistGradientBoostingRegressor(random_state=42)


## 3. Validation design

- **Hold-out test set (20%)**, created once by Person 2, stratified by category, `random_state=42`. Used only at the very end (P3-c).
- **5-fold cross-validation on the training set**, `KFold(5, shuffle=True, random_state=42)`. Each model is trained on 4 folds and scored on the 5th, five times. We report the **mean and the standard deviation** across folds: the mean says how good a model is, the std says how stable that result is.
- **CV sample:** to keep run time practical on a laptop, CV uses a stratified sample of **200,000** of the 940,325 training rows (each category keeps its share). The final tuned model in P3-c is trained on the full training set.

In [5]:
X_cv, y_cv = M.sample_training_rows(X_train, y_train, M.CV_SAMPLE)
print(f"CV rows: {len(X_cv):,} of {len(X_train):,}")
X_cv["category"].value_counts(normalize=True).rename("share")

CV rows: 200,000 of 940,325


category
Toys_and_Games              0.5939
Industrial_and_Scientific   0.2242
Musical_Instruments         0.1167
All_Beauty                  0.0652
Name: share, dtype: Float64

## 4. Cross-validation results

This cell takes a few minutes (RandomForest is the slowest). It saves the table to `reports/tables/cv_results.csv`, which Person 4's charts read.

In [6]:
cv_results = M.cross_validate_models(models, X_cv, y_cv)
cv_results.insert(1, "cv_rows", len(X_cv))
M.TABLES_DIR.mkdir(parents=True, exist_ok=True)
cv_results.to_csv(M.TABLES_DIR / "cv_results.csv", index=False)
cv_results

  Baseline (mean)        RMSE 0.5076 ± 0.0021   R2 -0.0000 ± 0.0000   (14s)
  Ridge                  RMSE 0.4599 ± 0.0022   R2 0.1793 ± 0.0015   (12s)
  RandomForest           RMSE 0.4536 ± 0.0023   R2 0.2016 ± 0.0025   (50s)
  HistGradientBoosting   RMSE 0.4515 ± 0.0025   R2 0.2089 ± 0.0028   (21s)


,model,cv_rows,rmse_mean,rmse_std,mae_mean,mae_std,r2_mean,r2_std,fit_time_s
0,HistGradientBoosting,200000,0.4515,0.0025,0.3347,0.0013,0.2089,0.0028,3.7306
1,RandomForest,200000,0.4536,0.0023,0.3357,0.0011,0.2016,0.0025,9.3075
2,Ridge,200000,0.4599,0.0022,0.3429,0.0012,0.1793,0.0015,2.0278
3,Baseline (mean),200000,0.5076,0.0021,0.3939,0.0011,-0.0000,0.0000,2.1863


In [7]:
# Improvement of each model over the baseline, in percent
base = cv_results.set_index("model").loc["Baseline (mean)"]
improvement = cv_results.set_index("model")[["rmse_mean", "mae_mean"]]
improvement = (1 - improvement / base[["rmse_mean", "mae_mean"]]) * 100
improvement.columns = ["RMSE reduction vs baseline (%)", "MAE reduction vs baseline (%)"]
improvement.round(1)

,RMSE reduction vs baseline (%),MAE reduction vs baseline (%)
model,,
HistGradientBoosting,11.1000,15.0000
RandomForest,10.6000,14.8000
Ridge,9.4000,12.9000
Baseline (mean),0.0000,0.0000


## 5. What the results mean

**Every model beats the baseline, so the listing features do carry signal.** The baseline's RMSE of 0.508 stars is simply the spread of ratings around their mean. HistGradientBoosting brings that down to **0.452 ± 0.003** stars (about 11% lower) and MAE from 0.394 to **0.335** stars (about 15% lower), explaining about **21% of the variation** in ratings (R² 0.209).

**The tree models beat the linear model.** Ridge reaches R² 0.179, while RandomForest (0.202) and HistGradientBoosting (0.209) do better. The gap between Ridge and the boosting model (0.008 RMSE) is about three fold standard deviations, so it is real, not noise. This means rating relationships are partly **non-linear**: for example, the effect of review volume or price is not a straight line.

**The two tree models are close.** HistGradientBoosting leads RandomForest by only about 0.002 RMSE, about one standard deviation, so on untuned settings they are nearly tied. HistGradientBoosting is chosen for tuning because it is as accurate and about **3× faster** to train (3.7 s vs 11.0 s per fold), which makes a search over settings practical.

**The results are stable.** Standard deviations across folds are around 0.002 stars for every model, so the ranking does not depend on how the data happened to be split.

**A modest R² is a finding, not a failure.** About 79% of the variation in ratings is *not* explained by price, category, brand scale, review volume and listing detail. Much of a product's rating is driven by things the listing does not describe, such as actual product quality, durability, shipping and customer expectations. An average error of about a third of a star matters on a scale where three quarters of products sit between 4.0 and 5.0.

*Numbers above come from the CV run on 200,000 training rows; they are reproduced exactly by re-running this notebook (fixed `random_state=42`).*

### Next (P3-c)

1. Tune HistGradientBoosting with `RandomizedSearchCV` (5-fold, RMSE scoring) on the training set.
2. Fit the tuned model on the full training set and score it **once** on the test set, reporting train and test metrics to check for overfitting.
3. Permutation feature importance, and errors by category.